In [ ]:
import torch
import torchvision.datasets
import torch.nn.functional as F


import transformers
print(f"Cuda available: {torch.cuda.is_available()}")

# Load the STL10 dataset (train and test splits)
data_train = torchvision.datasets.STL10(root='./data', split='train', download=True)
data_test = torchvision.datasets.STL10(root='./data', split='test', download=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


Check train/test split

In [ ]:
print(f"Train / Test Split Size  : {len(data_train):,} / {len(data_test):,}")

Label Encoding

In [ ]:
num_classes = len(data_train.classes)
print(f"Number of classes: {num_classes}")

In [ ]:
label_tensor = torch.tensor(data_train.labels, dtype=torch.long)
print(f"Label tensor shape: {label_tensor.shape}")

In [ ]:
onehot_labels = F.one_hot(label_tensor, num_classes=num_classes)
print(f"One-hot labels shape: {onehot_labels.shape}")

### Extraction of photo features with DINOv2

DINOv2 models produce high-performance visual features that can be directly employed with classifiers as simple as linear layers on a variety of computer vision tasks; these visual features are robust and perform well across domains without any requirement for fine-tuning. The models were pretrained on a dataset of 142 M images without using any labels or annotations.

https://github.com/facebookresearch/dinov2

https://dinov2.metademolab.com/

In [ ]:
model_name = "facebook/dinov2-base"
model = transformers.AutoModel.from_pretrained(model_name)

In [ ]:
#here we load processor for DinoV2
processor = transformers.AutoImageProcessor.from_pretrained(model_name)

In [ ]:
#and again check GPU
model = model.to(device)

model.eval()

In [ ]:
from tqdm import tqdm

In [ ]:
def getting_embeddings(images):
    #we make shuffle false because we want to keep the order of images and labels in the end
    data_loader = torch.utils.data.DataLoader(images, batch_size=16, shuffle=False)


    batch_embeddings = []

    with torch.no_grad():
        for batch in tqdm(data_loader):
            processed = processor(batch, return_tensors="pt", device=device) #make from 96x96 to 224x224
            outputs = model(processed['pixel_values']) #just go directly through the model

            normalized_emb = F.normalize(outputs.pooler_output, p=2, dim=1) #CLS token


            #dont forget to transfer them to CPU or we gonna face memory issues
            batch_embeddings.append(normalized_emb.cpu())

    return torch.cat(batch_embeddings, dim=0) #here we have smth like [amount of all pictures x 768]*

Lets extract embeddings from bost test/ train

In [ ]:
feature_train = getting_embeddings(data_train.data)
feature_test = getting_embeddings(data_test.data)

##Vector Indexing & K-NN Similarity Search (FAISS)

To build a scalable image retrieval engine, computing distance metrics via standard Python loops or naive matrix multiplications is highly inefficient ($O(N \times M \times D)$). Instead, this system integrates **FAISS (Facebook AI Similarity Search)**, a state-of-the-art library optimized for dense vector clustering and efficient similarity operations.

https://github.com/facebookresearch/faiss

### Concepts :
* **`faiss.IndexFlatL2`:** We initialize a flat index that computes the exact **Euclidean Distance (L2 Norm)** between the vector representations:
  $$d(x, y) = \sqrt{\sum_{i=1}^{n} (x_i - y_i)^2}$$
  Since a flat index performs an exhaustive search without structural compression, it provides a mathematically precise baseline for exact $K$-Nearest Neighbor ($K$-NN) classification.
* The 768-dimensional continuous embeddings extracted by DINOv2 from the training set are indexed into the gallery space. The unseen test embeddings then act as queries to retrieve the **Top-5 ($k=5$)** nearest visual neighbors.

In [ ]:
import faiss
import numpy as np
import matplotlib.pyplot as plt

# 1. Initialize GPU/CPU FAISS Index
embedding_dim = feature_train.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(feature_train.numpy()) # FAISS works natively with numpy/float32

def visualize_search_results(index, feature_test, data_test, data_train, k=5):
    """
    Selects random queries and visualizes the Top-K retrieved neighbors.
    """
    rand_indices = np.random.randint(low=0, high=len(data_test), size=k)
    query_embeddings = feature_test[rand_indices].numpy()

    distances, indices = index.search(query_embeddings, k)
    class_names = data_test.classes

    fig, axes = plt.subplots(k, k + 1, figsize=(16, 12), facecolor='white')
    plt.subplots_adjust(hspace=0.5, wspace=0.3)

    for row in range(k):
        # Plot Query Image
        query_img = data_test.data[rand_indices[row]].transpose(1, 2, 0)
        query_label = data_test.labels[rand_indices[row]]

        axes[row, 0].imshow(query_img)
        axes[row, 0].set_title(f"QUERY:\n{class_names[query_label].upper()}", color='darkblue', weight='bold', fontsize=9)
        axes[row, 0].axis('off')

        # Plot Retrieved Top-K Neighbors
        for col in range(1, k + 1):
            neighbor_idx = indices[row, col - 1]
            dist = distances[row, col - 1]
            neighbor_label = data_train.labels[neighbor_idx]
            neighbor_img = data_train.data[neighbor_idx].transpose(1, 2, 0)

            # Highlight errors in red, correct matches in green
            is_correct = neighbor_label == query_label
            title_color = 'green' if is_correct else 'red'

            axes[row, col].imshow(neighbor_img)
            axes[row, col].set_title(
                f"{class_names[neighbor_label]}\nDist: {dist:.2f}",
                color=title_color,
                fontsize=8
            )
            axes[row, col].axis('off')

    plt.suptitle("Vector Similarity Search Results (Top-5 Neighbors)", fontsize=14, weight='bold', y=0.95)
    plt.show()

# Run visualization
visualize_search_results(index, feature_test, data_test, data_train, k=5)

##Diagnostics via Retrieval Confusion Matrix

Evaluating a Content-Based Image Retrieval (CBIR) system requires metrics tailored to ranking and information retrieval rather than classic classification.

### Metrics:
1. **Mean Precision @ Top-K ($P@K$):** We compute the exact ratio of retrieved neighbors that share the true nominal label of the query image across the entire test subset (8,000 queries).
2. **Normalized Retrieval Confusion Matrix:** Relying solely on a global precision score can mask systematic model weaknesses. We construct a normalized confusion matrix visualized as a **Seaborn Heatmap**. This layout exposes cross-class confusion ratios, mapping how semantic categories interact within DINOv2's latent space.


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Execute Bulk Search for Evaluation
k_neighbors = 5
distances, indices = index.search(feature_test.numpy(), k_neighbors)

# 2. Compute Mean Precision @ Top-K
retrieved_labels = np.array(data_train.labels)[indices]
test_labels = np.array(data_test.labels).reshape(-1, 1)
matches = (retrieved_labels == test_labels)

mean_precision_at_k = np.mean(matches) * 100
print(f"Evaluation Metric - Mean Precision @ Top-{k_neighbors}: {mean_precision_at_k:.2f}%")

# 3. Generate Evaluation Analytics (Confusion Matrix)
def plot_retrieval_confusion_matrix(all_expected, all_retrieved, class_names):
    """
    Plots a beautiful heatmap showing which classes are being confused during retrieval.
    """
    cm = confusion_matrix(all_expected, all_retrieved, labels=range(len(class_names)))

    # Normalize by row (true classes)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(10, 8), facecolor='white')
    sns.heatmap(
        cm_normalized,
        annot=True,
        fmt=".2f",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Confusion Ratio'}
    )
    plt.title("Retrieval Confusion Matrix (Normalized)", fontsize=12, weight='bold', pad=20)
    plt.xlabel("Retrieved Class (Predicted Neighbor)", fontsize=10, labelpad=10)
    plt.ylabel("True Query Class", fontsize=10, labelpad=10)
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Flatten data for global error analysis across all retrieved neighbors
all_expected = np.repeat(data_test.labels, k_neighbors)
all_retrieved = retrieved_labels.flatten()
class_names = data_test.classes

plot_retrieval_confusion_matrix(all_expected, all_retrieved, class_names)

## Isolated Failure Mode Analysis (Error Auditing)

To understand the perceptual boundaries of the pre-trained Vision Transformer, this section introduces an automated **Error Auditor**. By filtering cases where $P@K \neq 100\%$, the system isolates and visualizes random retrieval mismatches side-by-side.

This diagnostic approach allows us to inspect edge cases and analyze whether the model prioritizes macro-environmental textures over specific anatomical definitions.

In [ ]:
def analyze_random_failure_mode(matches, indices, data_test, data_train):
    """
    Finds and isolates a random retrieval failure to inspect visually.
    """
    mismatch_rows, mismatch_cols = np.where(matches == False)

    if len(mismatch_rows) == 0:
        print("Zero mismatches found! Perfect retrieval score.")
        return

    print(f"Diagnostic: Total Retrieval Mismatches across Top-K slots: {len(mismatch_rows)}")

    # Pick a random error
    rand_idx = np.random.randint(5, len(mismatch_rows))
    query_idx = mismatch_rows[rand_idx]
    neighbor_rank = mismatch_cols[rand_idx]
    mismatched_train_idx = indices[query_idx, neighbor_rank]

    class_names = data_test.classes
    query_img = data_test.data[query_idx].transpose(1, 2, 0).astype(np.uint8)
    wrong_img = data_train.data[mismatched_train_idx].transpose(1, 2, 0).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), facecolor='white')

    axes[0].imshow(query_img)
    axes[0].set_title(f"Query Image\nTrue Label: {class_names[data_test.labels[query_idx]].upper()}", color='blue', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(wrong_img)
    axes[1].set_title(f"Mismatched Neighbor (Rank {neighbor_rank + 1})\nRetrieved Label: {class_names[data_train.labels[mismatched_train_idx]].upper()}", color='red', fontsize=10)
    axes[1].axis('off')

    plt.suptitle("Failure Mode Analysis Case Study", fontsize=12, weight='bold')
    plt.tight_layout()
    plt.show()

# Execute diagnostic
analyze_random_failure_mode(matches, indices, data_test, data_train)